In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent
con = sqlite3.connect(root / "data" / "db" / "tennis.db")

by_year = pd.read_sql("SELECT * FROM serve_advantage_by_year", con)
pooled = pd.read_sql("SELECT * FROM serve_advantage_pooled", con)
print(by_year.shape, pooled.shape)
print(by_year["computed_on"].iloc[0])

In [ ]:
def plot_serve_by_year(events, filename="serve_advantage_by_year.png"):
    fig, ax = plt.subplots(figsize=(9, 5.4))
    n = len(events)
    for i, name in enumerate(events):
        d = by_year[by_year["name"] == name].sort_values("file_year")
        ax.errorbar(d["file_year"] + (i - (n - 1) / 2) * 0.15, d["serve_excess_pp"], yerr=d["half_width"],
                    marker="o", linestyle="none", capsize=3, label=name)

    ax.axhline(0, color="grey", linewidth=1)
    ax.set_xticks(range(2015, 2027))
    ax.set_ylabel("Service points won above expectation\n(percentage points)")
    ax.set_title("Serve advantage by tournament and year, 2015-2026")
    ax.legend(loc="lower left")
    fig.text(0.01, 0.012,
             "Data source: TennisMyLife — https://stats.tennismylife.org. Vertical bars: approximate 95% bootstrap\n"
             "interval over matches. Baseline: other events with the same surface and roof, same players, +/- 1 year.",
             fontsize=7, color="grey")
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    (root / "charts").mkdir(exist_ok=True)
    fig.savefig(root / "charts" / filename, dpi=200)
    plt.show()

plot_serve_by_year(["Madrid Masters", "Monte Carlo", "Miami Masters"])

In [ ]:
ranking = pd.read_sql("SELECT * FROM elo_ranking_snapshots", con)


def plot_ranking(as_of, surface="Overall", top=10, filename=None):
    d = ranking[(ranking["as_of"] == as_of) & (ranking["surface"] == surface)].sort_values("rank").head(top)
    d = d.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 5.4))
    ax.scatter(d["rating"], range(len(d)), zorder=3)
    for i_, (r_, m_) in enumerate(zip(d["rating"], d["matches_52w"])):
        ax.text(r_ + 6, i_, f"{int(r_)}  ({int(m_)} matches, last 52 weeks)", va="center", fontsize=8)
    ax.set_yticks(range(len(d)))
    ax.set_yticklabels(d["name"])
    ax.set_xlim(d["rating"].min() - 40, d["rating"].max() + 190)
    ax.grid(axis="x", alpha=0.3)
    label = "overall rating" if surface == "Overall" else f"{surface.lower()} rating"
    ax.set_title(f"Top {top} by Elo {label}, as of {as_of}")
    fig.text(0.01, 0.012,
             "Data source: TennisMyLife — https://stats.tennismylife.org. Elo rating calculated from ATP tour-level\n"
             "main-draw results since 2000: our own rating, not the official ATP ranking, and not a prediction.",
             fontsize=7, color="grey")
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    (root / "charts").mkdir(exist_ok=True)
    fig.savefig(root / "charts" / (filename or f"elo_{surface.lower()}_{as_of}.png"), dpi=200)
    plt.show()


plot_ranking("2024-12-22")

In [ ]:
p26 = pd.read_sql("SELECT * FROM elo_period_2026", con).sort_values("rank")


def plot_year_view(d, top=10, filename="elo_overall_2026.png"):
    as_of = d["as_of"].iloc[0]
    d = d.head(top).iloc[::-1].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8.5, 5.6))
    for i_, row in d.iterrows():
        if pd.notna(row["elo_start"]):
            ax.plot([row["elo_start"], row["elo_end"]], [i_, i_], color="grey", linewidth=2, zorder=1)
            ax.scatter(row["elo_start"], i_, s=45, facecolor="white", edgecolor="grey", zorder=2)
            label = f"{int(row['elo_end'])}  ({int(row['change']):+d}, {int(row['matches_in_period'])} matches)"
        else:
            label = f"{int(row['elo_end'])}  (no rating on 1 Jan, {int(row['matches_in_period'])} matches)"
        ax.scatter(row["elo_end"], i_, s=55, zorder=3)
        ax.text(row["elo_end"] + 8, i_, label, va="center", fontsize=8)
    ax.set_yticks(range(len(d)))
    ax.set_yticklabels(d["name"])
    lows = d[["elo_start", "elo_end"]].min().min()
    ax.set_xlim(lows - 40, d["elo_end"].max() + 230)
    ax.grid(axis="x", alpha=0.3)
    ax.set_title(f"Elo overall rating, 1 Jan 2026 to {as_of}")
    fig.text(0.01, 0.012,
             "Data source: TennisMyLife — https://stats.tennismylife.org. Hollow dot: rating on 1 Jan 2026; filled dot: latest.\n"
             "Elo rating calculated from ATP tour-level main-draw results since 2000: our own rating, not the official ATP\n"
             "ranking, and not a prediction. Latest rating includes tournaments in progress.",
             fontsize=7, color="grey")
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    (root / "charts").mkdir(exist_ok=True)
    fig.savefig(root / "charts" / filename, dpi=200)
    plt.show()


plot_year_view(p26)

In [ ]:
ranking = tables["elo_ranking_snapshots"] = load("elo_ranking_snapshots")
latest_overall = ranking[ranking["surface"] == "Overall"]["as_of"].max()
print("drawing:", latest_overall)
plot_ranking(latest_overall, top=10, filename="elo_overall_2026.png")

In [ ]:
def load(table):
    try:
        return pd.read_sql(f"SELECT * FROM {table}", con)
    except Exception:
        return None


tables = {t: load(t) for t in ["serve_advantage_by_year", "serve_advantage_pooled",
                               "elo_ranking_snapshots", "player_season_cards"]}
for t, df in tables.items():
    print(t, "->", "MISSING" if df is None else df.shape)

ranking = tables["elo_ranking_snapshots"]